In [1]:
# Installation des bibliothèques nécessaires
!pip install -q transformers datasets accelerate sentencepiece sacrebleu evaluate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.6 MB/s eta 0:00:00


In [2]:
# Cellule 2: Imports et Configuration
import os
import torch
import numpy as np
import pandas as pd
from datasets import Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
from sklearn.model_selection import train_test_split # Import oublié
import matplotlib.pyplot as plt

# Configuration pour une meilleure reproductibilité
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Vérification du GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Device utilisé: {device}")
if torch.cuda.is_available():
    print(f" GPU: {torch.cuda.get_device_name(0)}")
    print(f" Mémoire disponible: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

 Device utilisé: cuda
 GPU: Tesla T4
 Mémoire disponible: 14.56 GB


In [3]:
original_dataset = pd.read_csv("dataset_final.csv")

In [4]:
print(original_dataset.head())
print(original_dataset.info())

           arb_TN_Arab                                eng_Latn
0              يا بنتي                          Oh my daughter
1  شرات جهاز موش نرمال  She bought a device that is not normal
2   ديجا هازاته جهازها                 Already took her device
3   خلت كان الدبش لهنا                 Left only the junk here
4    موش نرمال يا بنتي                 Not normal, my daughter
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 390117 entries, 0 to 390116
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   arb_TN_Arab  390117 non-null  object
 1   eng_Latn     390117 non-null  object
dtypes: object(2)
memory usage: 6.0+ MB
None


In [5]:
SAMPLE_SIZE = 100000
if len(original_dataset) > SAMPLE_SIZE:
    original_dataset = original_dataset.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
    print(f" Dataset réduit à {SAMPLE_SIZE} exemples pour un entraînement plus rapide.")
# ----------------------------------------------------------------------

# 1. Split du DataFrame pandas
train_df, test_df = train_test_split(
    original_dataset,
    test_size=0.2,
    random_state=SEED
)

# 2. Fonction pour créer un dataset bidirectionnel
def create_bidirectional_dataset(df):
    eng_to_tn = {
        'source': df['eng_Latn'],
        'target': df['arb_TN_Arab'],
        'direction': ['eng2tn'] * len(df)
    }

    tn_to_eng = {
        'source': df['arb_TN_Arab'],
        'target': df['eng_Latn'],
        'direction': ['tn2eng'] * len(df)
    }

    dataset_eng2tn = Dataset.from_dict(eng_to_tn)
    dataset_tn2eng = Dataset.from_dict(tn_to_eng)

    bidirectional_dataset = concatenate_datasets(
        [dataset_eng2tn, dataset_tn2eng]
    )

    return bidirectional_dataset.shuffle(seed=SEED)

# 3. Création des datasets bidirectionnels
print(" Création du dataset bidirectionnel...")

train_bidirectional = create_bidirectional_dataset(train_df)
test_bidirectional = create_bidirectional_dataset(test_df)

print(" Dataset bidirectionnel créé:")
print(f"   - Train: {len(train_bidirectional)} exemples")
print(f"   - Test: {len(test_bidirectional)} exemples")
print(f"   - Colonnes: {train_bidirectional.column_names}")

 Dataset réduit à 100000 exemples pour un entraînement plus rapide.
 Création du dataset bidirectionnel...
 Dataset bidirectionnel créé:
   - Train: 160000 exemples
   - Test: 40000 exemples
   - Colonnes: ['source', 'target', 'direction']


In [6]:
# Configuration
MODEL_NAME = "facebook/nllb-200-distilled-600M"
MAX_LENGTH = 128

print(f" Chargement du modèle: {MODEL_NAME}")
print(f" Mode: BIDIRECTIONNEL (eng ↔ tunisien)")

# Chargement du tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

LANG_CODES = {
    'eng': 'eng_Latn',  # Anglais
    'tn': 'arb_Arab'    # Tunisien
}
# Récupérer les IDs des tokens de langue pour plus tard
LANG_IDS = {
    'eng': tokenizer.convert_tokens_to_ids('eng_Latn'),
    'tn': tokenizer.convert_tokens_to_ids('arb_Arab')
}
print(f" Codes de langue:")
print(f"   - Anglais: {LANG_CODES['eng']} (ID: {LANG_IDS['eng']})")
print(f"   - Tunisien: {LANG_CODES['tn']} (ID: {LANG_IDS['tn']})")

# S'assurer que le tokenizer a un token de padding
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Chargement du modèle
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
# Activer le gradient checkpointing pour économiser la mémoire
model.gradient_checkpointing_enable()
print(" Modèle chargé avec succès")

 Chargement du modèle: facebook/nllb-200-distilled-600M
 Mode: BIDIRECTIONNEL (eng ↔ tunisien)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

 Modèle chargé avec succès


In [7]:
# Cellule 5: Tokenisation du Dataset
def preprocess_bidirectional(examples):
    # Tokenisation des sources
    model_inputs = tokenizer(
        examples['source'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False # On laisse le data collator gérer le padding dynamique
    )

    # Tokenisation des cibles
    labels = tokenizer(
        text_target=examples['target'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False # Idem
    )

    model_inputs["labels"] = labels["input_ids"]

    # Définition dynamique du forced_bos_token_id
    forced_bos_token_ids = []
    for direction in examples['direction']:
        if direction == 'eng2tn':
            forced_bos_token_ids.append(LANG_IDS['tn'])
        else: # tn2eng
            forced_bos_token_ids.append(LANG_IDS['eng'])

    model_inputs["forced_bos_token_id"] = forced_bos_token_ids

    return model_inputs

# Tokenisation du dataset
print(" Tokenisation du dataset...")
tokenized_train = train_bidirectional.map(
    preprocess_bidirectional,
    batched=True,
    remove_columns=train_bidirectional.column_names,
    desc="Tokenisation train"
)

tokenized_test = test_bidirectional.map(
    preprocess_bidirectional,
    batched=True,
    remove_columns=test_bidirectional.column_names,
    desc="Tokenisation test"
)

print(" Tokenisation terminée")
print(f"   - Train: {len(tokenized_train)} exemples")
print(f"   - Test: {len(tokenized_test)} exemples")

 Tokenisation du dataset...


Tokenisation train:   0%|          | 0/160000 [00:00<?, ? examples/s]

Tokenisation test:   0%|          | 0/40000 [00:00<?, ? examples/s]

 Tokenisation terminée
   - Train: 160000 exemples
   - Test: 40000 exemples


In [8]:
# Configuration LoRA optimisée pour traduction bidirectionnelle
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
    target_modules=[
        "q_proj",
        "v_proj",
        "k_proj",
        "out_proj",
    ],
    inference_mode=False,
)

# Application de LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 9,437,184 || all params: 1,411,575,808 || trainable%: 0.6686


In [9]:
# Cellule 7: Configuration des Métriques
# Métrique BLEU
bleu_metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    # Remplacer les -100 par le token de pad
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Décodage des prédictions et des références
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Post-processing simple
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    # Calcul du BLEU score
    result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "bleu": result["score"],
        "bleu_precisions": np.mean(result["precisions"])
    }

print(" Métriques configurées")

 Métriques configurées


In [12]:
# Cellule 8: Configuration du Trainer (CORRIGÉE)
OUTPUT_DIR   = "./nllb-tunisian"
FINAL_MODEL_PATH = os.path.join(OUTPUT_DIR, "final_model")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    # Évaluation et sauvegarde plus fréquentes sur un petit dataset
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    # Hyperparamètres d'entraînement
    learning_rate=5e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3, # Réduit de 8 à 3, suffisant pour LoRA sur ce dataset
    warmup_steps=200,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    # Génération pour l'évaluation
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    generation_num_beams=4,
    # Optimisations
    fp16=True,
    dataloader_num_workers=2,
    gradient_checkpointing=True,
    # Logging - CORRECTION ICI
    logging_dir=f"{OUTPUT_DIR}/logs",  # Ceci est déprécié mais fonctionne encore
    logging_steps=100,
    logging_first_step=True,
    report_to="none",
    # Sélection du meilleur modèle
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    # Optimiseur
    optim="adamw_torch",
)

# Data collator (pour le padding dynamique)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

# Création du trainer - CORRECTION: retirer le paramètre tokenizer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    # tokenizer=tokenizer,  # <- SUPPRIMER CETTE LIGNE
)

# Pour sauvegarder le tokenizer avec le modèle, on le fait manuellement après l'entraînement
# ou on peut l'attacher au trainer après création:
trainer.tokenizer = tokenizer

print("✅ Trainer créé")
print(f"\n🎯 Début de l'entraînement...")
total_steps = len(tokenized_train) * training_args.num_train_epochs // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
print(f"   Total steps: {total_steps}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Trainer créé

🎯 Début de l'entraînement...
   Total steps: 30000


In [ ]:
# Nettoyage mémoire
torch.cuda.empty_cache()

# Entraînement
train_result = trainer.train()

# Sauvegarde finale du modèle et du tokenizer
trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)
print(f" Entraînement terminé. Modèle final sauvegardé dans {FINAL_MODEL_PATH}")

Step,Training Loss,Validation Loss


In [ ]:
# Cellule 10: Évaluation Finale et Test
print(" Évaluation finale sur le jeu de test...")
eval_results = trainer.evaluate()
print(f" BLEU Score Final: {eval_results['eval_bleu']:.2f}")

# Fonction de traduction
def translate(text, source_lang='eng', target_lang='tn'):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
    forced_bos_token_id = LANG_IDS[target_lang]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_length=MAX_LENGTH,
            num_beams=4,
            early_stopping=True
        )

    translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated_text

# Tests
print("\n Tests de traduction:")
print("Anglais -> Tunisien:")
print(f"  'Hello, how are you?' -> '{translate('Hello, how are you?', 'eng', 'tn')}'")
print(f"  'The weather is nice today.' -> '{translate('The weather is nice today.', 'eng', 'tn')}'")
print("\nTunisien -> Anglais:")
print(f"  'شنو حالك؟' -> '{translate('شنو حالك؟', 'tn', 'eng')}'")
print(f"  'الجو باهي اليوم.' -> '{translate('الجو باهي اليوم.', 'tn', 'eng')}'")